In [1]:
%logstart -o notebook_log.txt append

Activating auto-logging. Current session state plus future input saved.
Filename       : notebook_log.txt
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : False
State          : active


In [ ]:
import sys
from pathlib import Path
print(Path.cwd())
import chromadb
from vorstellungsgesprach.notebook import prepare_jobs
from vorstellungsgesprach.key_terms import analyze_jobs
from vorstellungsgesprach.chunck import build_documents
from vorstellungsgesprach.utils import load_data, add_language_metadata,remove_duplicate_jobs
from vorstellungsgesprach import conf
from vorstellungsgesprach import embeddings
from vorstellungsgesprach import evaluation
from vorstellungsgesprach import conf, rag
from vorstellungsgesprach.models import list_available_chat_models
from google import genai
import json
import os




/Users/eli/vorstellungsgesprach/notebook
/Users/eli/vorstellungsgesprach/notebook


In [3]:
#defining API key for GenAI
client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [4]:
jobs = prepare_jobs("../data/raw/job.json")
print(len(jobs))

40


In [9]:
sample_jobs = jobs[:5]
analyzed_jobs = analyze_jobs(sample_jobs)


Analysiere Stelle 1/5: consultant data science & ml engineering in stuttgart
Analysiere Stelle 2/5: (senior) data scientist
Analysiere Stelle 3/5: senior data scientist
Analysiere Stelle 4/5: machine learning engineer*
Analysiere Stelle 5/5: data & analytics engineer


In [13]:
print(analyzed_jobs[0].keys())
print(json.dumps(analyzed_jobs[0]["competencies"], ensure_ascii=False, indent=2))

dict_keys(['id', 'description', 'title', 'company', 'detected_language', 'language_confidence', 'description_lower', 'canonical_role', 'seniority', 'competencies'])
[
  {
    "canonical_name": "Python",
    "original_expression": "Python",
    "category": "programming_language",
    "requirement_level": "good",
    "context": "Gute Kenntnisse in Python und SQL sowie erste praktische Erfahrungen mit Data-Science- und Machine-Learning-Frameworks wie beispielsweise scikit-learn, LightGBM oder vergleichbaren Technologien"
  },
  {
    "canonical_name": "SQL",
    "original_expression": "SQL",
    "category": "programming_language",
    "requirement_level": "good",
    "context": "Gute Kenntnisse in Python und SQL sowie erste praktische Erfahrungen mit Data-Science- und Machine-Learning-Frameworks wie beispielsweise scikit-learn, LightGBM oder vergleichbaren Technologien"
  },
  {
    "canonical_name": "scikit-learn",
    "original_expression": "scikit-learn",
    "category": "tool",
    "r

In [10]:
with open("../data/processed/key_terms_sample.json", "w", encoding="utf-8") as f:
    json.dump(analyzed_jobs, f, ensure_ascii=False, indent=2)

In [14]:
from collections import defaultdict

def build_keyword_index(extractions):
    keyword_to_jobs = defaultdict(list)
    keyword_to_contexts = defaultdict(set)

    for entry in extractions:
        for comp in entry["competencies"]:
            term_key = comp["canonical_name"].strip().lower()
            if not term_key:
                continue

            keyword_to_jobs[term_key].append({
                "title": entry.get("title"),
                "company": entry.get("company"),
            })
            keyword_to_contexts[term_key].add(comp["context"])

    return {
        "keyword_to_jobs": dict(keyword_to_jobs),
        "keyword_to_contexts": {k: sorted(v) for k, v in keyword_to_contexts.items()},
    }

keyword_index = build_keyword_index(analyzed_jobs)

with open("../data/processed/keyword_index_sample.json", "w", encoding="utf-8") as f:
    json.dump(keyword_index, f, ensure_ascii=False, indent=2)

print(f"Keywords únicas nas 5 vagas: {len(keyword_index['keyword_to_jobs'])}")
print(list(keyword_index["keyword_to_jobs"].keys()))

Keywords únicas nas 5 vagas: 72
['python', 'sql', 'scikit-learn', 'lightgbm', 'machine learning', 'mlops', 'microsoft azure', 'amazon web services', 'google cloud platform', 'german', 'english', 'deep learning', 'generative ai', 'aws', 'deployment', 'monitoring', 'rag', 'prompt engineering', 'agentic ai systems', 'statistics', 'mathematics', 'computer science', 'business informatics', 'data science', 'informatik', 'mathematik', 'statistik', 'ingenieurwissenschaften', 'datenanalyse', 'pandas', 'apache spark', 'hugging face', 'mosaic ai', 'pytorch', 'tensorflow', 'llm', 'statistische methoden', 'datenvisualisierung', 'databricks', 'gcp', 'kommunikationsfähigkeit', 'teamfähigkeit', 'ml pipelines', 'cloud', 'edge computing', 'nosql', 'rest api', 'version control', 'code review', 'unit testing', 'integration testing', 'performance testing', 'containerization', 'ci/cd', 'stakeholder management', 'computer vision', 'typescript', 'vue.js', 'react', 'tensorrt', 'onnx', 'cuda', 'cudnn', 'datenmo

In [15]:
test_keyword = "python"

print("=== VAGAS ===")
for job in keyword_index["keyword_to_jobs"].get(test_keyword, []):
    print(f"- {job['title']} @ {job['company']}")

print("\n=== CONTEXTOS ===")
for ctx in keyword_index["keyword_to_contexts"].get(test_keyword, []):
    print(f"- {ctx}\n")

=== VAGAS ===
- consultant data science & ml engineering in stuttgart @ deloitte
- (senior) data scientist @ dräxlmaier group
- senior data scientist @ fendt
- machine learning engineer* @ tomra
- data & analytics engineer @ hermes group

=== CONTEXTOS ===
- Analytische Methoden: Idealerweise verfügst du über Kenntnisse in Python, Statistik oder Advanced Analytics und bringst diese bei der Entwicklung datengetriebener Lösungen ein.

- Fundierte Kenntnisse in Python sowie in Cloud-Plattformen wie AWS oder Microsoft Azure

- Fundierte Kenntnisse in Python, SQL und modernen Data-Science- und Machine-Learning-Frameworks wie Pandas, Spark, scikit-learn, Hugging Face, Mosaic AI, PyTorch oder TensorFlow

- Gute Kenntnisse in Python und SQL sowie erste praktische Erfahrungen mit Data-Science- und Machine-Learning-Frameworks wie beispielsweise scikit-learn, LightGBM oder vergleichbaren Technologien

- Sehr gute Kenntnisse in Python



In [ ]:
def chunk_description(text: str, max_chars: int = 4000, overlap: int = 200) -> list[str]:
    """Quebra a descrição em pedaços se for muito longa; senão, retorna como está."""
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end])
        start = end - overlap  # overlap evita cortar contexto no meio de uma frase importante

    return chunks


def build_documents(jobs: list[dict]) -> list[dict]:
    """Transforma cada vaga em um ou mais documentos, dependendo do tamanho."""
    documents = []
    for job in jobs:
        description_chunks = chunk_description(job["description"])

        for i, chunk in enumerate(description_chunks):
            text = f"Titel: {job['title']}\nUnternehmen: {job['company']}\n\n{chunk}"
            documents.append({
                "id": f"{job['id']}_chunk{i}" if len(description_chunks) > 1 else job["id"],
                "text": text,
                "metadata": {
                    "job_id": job["id"],
                    "title": job["title"],
                    "company": job["company"],
                    "chunk_index": i,
                    "total_chunks": len(description_chunks),
                },
            })
    return documents


documents = build_documents(jobs)
print(f"Total de vagas: {len(jobs)}")
print(f"Total de documentos (após chunking): {len(documents)}")

Total de vagas: 40
Total de documentos (após chunking): 56


In [ ]:
#create documents for embedding
texts = [doc["text"] for doc in documents]
vectors = embeddings.embed_texts(texts)

print(f"Embeddings gerados: {len(vectors)}")
print(f"Dimensão de cada vetor: {len(vectors[0])}")

Exception: No such file or directory (os error 2)

In [ ]:
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="vagas_ti")

collection.add(
    ids=[doc["id"] for doc in documents],
    embeddings=vectors,
    documents=[doc["text"] for doc in documents],
    metadatas=[doc["metadata"] for doc in documents],
)

print(f"Documentos indexados: {collection.count()}")

NameError: name 'vectors' is not defined

In [ ]:
#check models available
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "vagas_ti"


chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

question = "Welche Stellen erfordern Erfahrung mit Python?"

gemini_client = genai.Client(api_key=conf.GEMINI_API_KEY)
available_models = list_available_chat_models(gemini_client)

results = []

for model_name in available_models:
    try:
        result = rag.answer(
            collection=collection,
            query=question,
            model=model_name,
            n_results=20,
        )

        results.append(result)

        print(f"\n{'=' * 80}")
        print(f"Modelo: {result['model']}")
        print(f"{'=' * 80}")
        print(result["answer"])

    except Exception:
        continue

if results:
    print("\nFontes recuperadas:")

    for index, source in enumerate(results[0]["sources"], start=1):
        metadata = source["metadata"]

        print(
            f"{index}. {metadata.get('title', 'Sem título')} "
            f"@ {metadata.get('company', 'Sem empresa')} "
            f"(distância: {source['distance']:.3f})"
        )
else:
    print("Nenhum modelo produziu uma resposta.")

Nenhum modelo produziu uma resposta.


In [ ]:
judge_model = available_models[0]
judged_results = []

for result in results:
    context = "\n\n".join(
        source["text"]
        for source in result["sources"]
    )

    try:
        scores = evaluation.judge_answer(
            query=result["query"],
            answer=result["answer"],
            context=context,
            judge_model=judge_model,
        )

        judged_results.append(
            {
                **result,
                **scores,
            }
        )

    except Exception:
        continue


best = sorted(
    judged_results,
    key=lambda result: result["overall"] or 0,
    reverse=True,
)


for result in best:
    print(
        f"{result['model']} | "
        f"faithfulness={result['faithfulness']} | "
        f"helpfulness={result['helpfulness']} | "
        f"overall={result['overall']}"
    )


winner = best[0]

print(
    f"\nMelhor modelo: {winner['model']} "
    f"(nota: {winner['overall']})"
)

IndexError: list index out of range